# Множинна регресія та попереднє опрацювання даних
## Завдання 1
Побудуйте модель лінійної регресії на кількох ознаках.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline

# Згенеруємо дані та розділимо їх
X, y = make_regression(n_samples=2000, n_features=4, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X_train, y_train)
r2_1 = r2_score(y_test, model1.predict(X_test))

print(f"R² на тестовій вибірці: {r2_1:.5f}")
coef_df = pd.DataFrame({
    'Ознака': ['x1', 'x2', 'x3', 'x4'],
    'Коефіцієнт': model1.coef_
})
display(coef_df)

R² на тестовій вибірці: 0.99372


,Ознака,Коефіцієнт
0,x1,3.607749
1,x2,46.862862
2,x3,91.884679
3,x4,39.636486


## Завдання 2
Перевірте вплив масштабування на роботу лінійної регресії.

In [2]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model2 = LinearRegression()
model2.fit(X_train_scaled, y_train)
r2_2 = r2_score(y_test, model2.predict(X_test_scaled))

task2_df = pd.DataFrame({
    'Масштабування': ['Ні', 'StandardScaler'],
    'R²': [r2_1, r2_2]
})
display(task2_df)

,Масштабування,R²
0,Ні,0.993724
1,StandardScaler,0.993724


## Завдання 3
Перевірте, як заповнення пропусків впливає на модель.

In [3]:
X_missing = X.copy()
np.random.seed(42)
# Зробимо пропущеними близько 10% значень в першій ознаці (x1)
missing_mask = np.random.rand(X_missing.shape[0]) < 0.1
X_missing[missing_mask, 0] = np.nan

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_missing, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train_m)
X_test_imp = imputer.transform(X_test_m)

model3 = LinearRegression()
model3.fit(X_train_imp, y_train_m)
r2_3 = r2_score(y_test_m, model3.predict(X_test_imp))

task3_df = pd.DataFrame({
    'Сценарій': ['Без пропусків', 'Після ім\'ютації'],
    'R²': [r2_1, r2_3]
})
display(task3_df)

,Сценарій,R²
0,Без пропусків,0.993724
1,Після ім'ютації,0.993732


## Завдання 4
Оцініть стійкість моделі через крос-валідацію.

In [4]:
# Нові дані з більшим шумом
X_new, y_new = make_regression(n_samples=2000, n_features=4, noise=15, random_state=42)

# Створюємо pipeline, щоб стандартизація відбувалась всередині кожного фолду коректно
pipeline = make_pipeline(StandardScaler(), LinearRegression())

cv_scores = cross_val_score(pipeline, X_new, y_new, cv=5, scoring='r2')

print(f"Середне значення R²: {np.mean(cv_scores):.5f}")
print(f"Стандартне відхилення R²: {np.std(cv_scores):.5f}")

Середне значення R²: 0.98238
Стандартне відхилення R²: 0.00121
